In [1]:
from google.colab import files
uploaded = files.upload()

Saving car.mp4 to car.mp4


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow


def calculate_speed(flow, scale_factor, fps):
    """
    Estimate speed in km/h from optical flow vectors.
    """
    if flow.size == 0:
        return 0

    magnitudes = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
    avg_magnitude = np.mean(magnitudes)

    speed_m_per_s = avg_magnitude * scale_factor * fps
    speed_km_per_h = speed_m_per_s * 3.6

    return speed_km_per_h


def detect_vehicles(frame, fg_mask):

    # Threshold mask (important fix)
    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    vehicle_contours = []

    for contour in contours:
        if cv2.contourArea(contour) > 1500:

            x, y, w, h = cv2.boundingRect(contour)

            if h == 0:
                continue

            aspect_ratio = w / float(h)

            # fixed condition
            if 0.3 < aspect_ratio < 1.5:
                vehicle_contours.append((x, y, w, h))

    return vehicle_contours


def draw_bounding_box(frame, vehicles, flow, scale_factor, fps):

    for (x, y, w, h) in vehicles:

        roi_flow = flow[y:y + h, x:x + w]

        speed = calculate_speed(roi_flow, scale_factor, fps)

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2,
        )

        cv2.putText(
            frame,
            f"{speed:.1f} km/h",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )


def main(video_path, scale_factor):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)   # auto fps

    ret, prev_frame = cap.read()

    if not ret:
        print("Cannot read video")
        return

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    background_subtractor = cv2.createBackgroundSubtractorMOG2()

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optical flow
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray,
            gray,
            None,
            0.5,
            3,
            15,
            3,
            5,
            1.2,
            0,
        )

        # Background subtraction (use frame not gray)
        fg_mask = background_subtractor.apply(frame)

        vehicles = detect_vehicles(frame, fg_mask)

        draw_bounding_box(
            frame,
            vehicles,
            flow,
            scale_factor,
            fps,
        )

        cv2_imshow(frame)

        prev_gray = gray.copy()

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":

    video_path = r"car.mp4"

    scale_factor = 0.05   # meter per pixel

    main(video_path, scale_factor)